In [1]:
# ============================================================
# SMART LECTURE NOTES GENERATOR - WEB APPLICATION
# Google Colab - Single Code
# ============================================================

!pip -q install gradio SpeechRecognition pydub nltk

import gradio as gr
import speech_recognition as sr
import nltk
import re
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from pydub import AudioSegment

# Download NLTK data
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

# ------------------------------------------------------------
# MAIN FUNCTION
# ------------------------------------------------------------

def generate_notes(audio_file):

    if audio_file is None:
        return "Please upload an audio file.", "", ""

    try:

        # ----------------------------------------------------
        # Convert audio to WAV
        # ----------------------------------------------------

        if audio_file.lower().endswith(".mp3"):

            audio = AudioSegment.from_mp3(audio_file)
            wav_file = "lecture.wav"
            audio.export(wav_file, format="wav")

        else:
            wav_file = audio_file

        # ----------------------------------------------------
        # Speech to Text
        # ----------------------------------------------------

        recognizer = sr.Recognizer()

        with sr.AudioFile(wav_file) as source:

            audio_data = recognizer.record(source)

        text = recognizer.recognize_google(audio_data)

        # ----------------------------------------------------
        # Remove unnecessary words
        # ----------------------------------------------------

        unnecessary_words = [
            "um",
            "uh",
            "hmm",
            "okay",
            "ok",
            "like",
            "actually",
            "basically",
            "you know",
            "right"
        ]

        clean_text = text

        for word in unnecessary_words:

            clean_text = re.sub(
                r"\b" + re.escape(word) + r"\b",
                "",
                clean_text,
                flags=re.IGNORECASE
            )

        clean_text = re.sub(
            r"\s+",
            " ",
            clean_text
        ).strip()

        # ----------------------------------------------------
        # Keyword Extraction
        # ----------------------------------------------------

        stop_words = set(stopwords.words("english"))

        words = word_tokenize(clean_text.lower())

        important_words = []

        for word in words:

            if (
                word.isalpha()
                and word not in stop_words
                and len(word) > 3
            ):
                important_words.append(word)

        frequency = Counter(important_words)

        keywords = [
            word
            for word, count in frequency.most_common(10)
        ]

        keyword_output = "\n".join(
            [f"• {word}" for word in keywords]
        )

        # ----------------------------------------------------
        # Summary Generation
        # ----------------------------------------------------

        sentences = sent_tokenize(clean_text)

        sentence_scores = {}

        for sentence in sentences:

            score = 0

            sentence_words = word_tokenize(
                sentence.lower()
            )

            for word in sentence_words:

                if word in frequency:
                    score += frequency[word]

            sentence_scores[sentence] = score

        summary_sentences = sorted(
            sentence_scores,
            key=sentence_scores.get,
            reverse=True
        )[:3]

        summary = " ".join(summary_sentences)

        # ----------------------------------------------------
        # Final Notes
        # ----------------------------------------------------

        notes = f"""
# SMART LECTURE NOTES

## Lecture Text

{clean_text}

## Important Keywords

{keyword_output}

## Summary

{summary}

---

Notes generated automatically from the uploaded lecture.
"""

        return clean_text, keyword_output, summary

    except sr.UnknownValueError:

        return (
            "Speech could not be understood.",
            "",
            ""
        )

    except Exception as e:

        return (
            f"Error: {str(e)}",
            "",
            ""
        )


# ============================================================
# APPLICATION INTERFACE
# ============================================================

with gr.Blocks(title="Smart Lecture Notes Generator") as app:

    gr.Markdown(
        """
        # 🎓 Smart Lecture Notes Generator

        ### Convert teacher's lecture audio into useful study notes

        **Audio → Speech-to-Text → Clean Text → Keywords → Summary**
        """
    )

    with gr.Row():

        audio_input = gr.Audio(
            sources=["upload"],
            type="filepath",
            label="🎤 Upload Teacher's Audio"
        )

    generate_button = gr.Button(
        "📝 Generate Lecture Notes",
        variant="primary"
    )

    gr.Markdown("## 📄 Lecture Text")

    text_output = gr.Textbox(
        label="Speech-to-Text / Cleaned Text",
        lines=8
    )

    with gr.Row():

        keyword_output = gr.Textbox(
            label="🔑 Important Keywords",
            lines=8
        )

        summary_output = gr.Textbox(
            label="📚 Lecture Summary",
            lines=8
        )

    generate_button.click(
        fn=generate_notes,
        inputs=audio_input,
        outputs=[
            text_output,
            keyword_output,
            summary_output
        ]
    )

    gr.Markdown(
        """
        ---
        ### Features

        ✅ Upload teacher audio
        ✅ Speech-to-text conversion
        ✅ Remove unnecessary words
        ✅ Extract important keywords
        ✅ Generate lecture summary
        ✅ Display study notes

        **Text and Speech Analysis Project**
        """
    )


# Launch application
app.launch(share=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 72.1 MB/s eta 0:00:00
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1b1fd07e2b66a38fcc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
